# Wildlife Network starter notebook

Arrival dates, weekly records and observation effort from [Wildlife Network](https://wlnw.org). Methods: https://wlnw.org/methods. API: https://wlnw.org/developers.

Cells are 5 degree squares named by their south-west corner; weeks start on Monday. Cite the data release DOI 10.5281/zenodo.23193453 and the GBIF datasets it lists.

In [ ]:
import io, zipfile, urllib.request
import pandas as pd

BASE = 'https://wlnw.org'

## 1. Arrival dates for one species (API)

Each row is one 5 degree cell: arrival week with a 90% interval, and support (share of 40 redraws that found a season; below 0.5 is weak).

In [ ]:
species = 'Hirundo rustica'
arr = pd.read_csv(f"{BASE}/api/v1/arrivals?species={urllib.parse.quote(species)}&format=csv", comment='#', parse_dates=['arrivalWeek', 'arrivalLow', 'arrivalHigh', 'peakWeek'])
arr = arr[arr.arrivalSupport >= 0.5]
arr[['region', 'cellLat', 'cellLon', 'arrivalWeek', 'arrivalLow', 'arrivalHigh', 'arrivalSupport']].sort_values('arrivalWeek').head(20)

## 2. Arrival week against latitude

A simple check of the northward progression within one region.

In [ ]:
eu = arr[arr.region == 'Europe'].copy()
eu['doy'] = eu.arrivalWeek.dt.dayofyear
ax = eu.plot.scatter(x='cellLat', y='doy', title=f'{species}: arrival day of year by latitude, Europe')
print('days per degree north:', round(eu[['cellLat', 'doy']].cov().iloc[0, 1] / eu.cellLat.var(), 2))

## 3. The open data release (GBIF-derived, CC BY 4.0)

Weekly records per species and cell with the effort behind them, so you can compute shares yourself.

In [ ]:
ZIP = 'https://github.com/wlnw-org/wildnetwork/releases/download/data-2026-10-25/wildnetwork-open-2026-10-25.zip'
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(ZIP).read()))
name = lambda f: next(n for n in z.namelist() if n.endswith(f))
weekly = pd.read_csv(z.open(name('weekly_records.csv')), parse_dates=['week'])
effort = pd.read_csv(z.open(name('effort.csv')), parse_dates=['week'])
weekly.head()

## 4. Effort-corrected share for one species in one cell

Share = records of the species / records of its whole class in the same cell and week (the effort).

In [ ]:
cell = weekly[(weekly.scientific_name == species) & (weekly.cell_lat == 50) & (weekly.cell_lon == 0)]
m = cell.merge(effort, on=['week', 'cell_lat', 'cell_lon', 'class'], suffixes=('', '_all'))
m['per_1000'] = 1000 * m.records / m.records_all
m.plot(x='week', y='per_1000', title=f'{species}, cell 50N 0E, records per 1,000 bird records')

## Next

- Draw a study area and get a dashboard and weekly digest: https://wlnw.org/studies
- Review arrival estimates you know well (ORCID sign-in): https://wlnw.org/research#review
- Questions or a study idea: https://wlnw.org/research#propose